# Tarea 3 — Clasificación de Fashion MNIST con PyTorch

- **Nombre:** Máximo Mujica
- **Cédula:** 30727257

Este notebook permite **crear, entrenar, probar, guardar y cargar** redes neuronales para
clasificar Fashion MNIST (10 clases originales o 4 clases simplificadas).

> En esta tarea "convolucional" **no** significa capas `Conv2d`: es una arquitectura de
> *embudo* donde el número de neuronas baja linealmente de 784 hasta la capa de salida.

Para usarlo: **Restart & Run All** y el notebook termina en el menú interactivo de la última celda.

In [1]:
import copy
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NOTEBOOK_NUMBER = 1          # N in pytorch_fashion_N_*.pth
DATA_DIR = Path("data")      # already in .gitignore
INPUT_SIZE = 784             # 28 x 28 pixels
BATCH_SIZE = 128
VALIDATION_SIZE = 5000       # held out from the training set (used by early stopping)

print(f"Device: {DEVICE}")

Device: cpu


## 1. Datos y clases

Los píxeles entran en el rango `[0, 1]` (`transforms.ToTensor()`), **sin** `Normalize` externo.
El conjunto de entrenamiento se divide en 55 000 imágenes de entrenamiento y 5 000 de validación
(la validación solo se usa para *early stopping* y para vigilar el sobreajuste; el conjunto de
**prueba** nunca se usa para entrenar).

Mapeo de clases simplificadas (orden de salida: `0` Top, `1` Bottom, `2` Footwear, `3` Bag):

| Simplificada | Clases originales |
|---|---|
| Top (0) | T-shirt [0], Pullover [2], Coat [4], Shirt [6] |
| Bottom (1) | Trouser [1], Dress [3] |
| Footwear (2) | Sandal [5], Sneaker [7], Ankle boot [9] |
| Bag (3) | Bag [8] |

In [2]:
ORIGINAL_CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
SIMPLIFIED_CLASSES = ["Top", "Bottom", "Footwear", "Bag"]

# Index = original label, value = simplified label.
ORIGINAL_TO_SIMPLIFIED = torch.tensor([0, 1, 0, 1, 0, 2, 0, 2, 3, 2])

CLASS_NAMES = {"original": ORIGINAL_CLASSES, "simplified": SIMPLIFIED_CLASSES}
NUM_OUTPUTS = {"original": 10, "simplified": 4}


def _dataset_to_tensors(dataset) -> Tuple[torch.Tensor, torch.Tensor]:
    """Loads a whole torchvision dataset into two tensors (images in [0, 1], labels)."""
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    images, labels = next(iter(loader))
    return images, labels


def load_fashion_mnist(root: Path = DATA_DIR):
    transform = transforms.ToTensor()
    train_set = datasets.FashionMNIST(root=str(root), train=True, download=True, transform=transform)
    test_set = datasets.FashionMNIST(root=str(root), train=False, download=True, transform=transform)
    x_train, y_train = _dataset_to_tensors(train_set)
    x_test, y_test = _dataset_to_tensors(test_set)
    return x_train, y_train, x_test, y_test


X_TRAIN_FULL, Y_TRAIN_FULL, X_TEST, Y_TEST = load_fashion_mnist()

_perm = torch.randperm(len(X_TRAIN_FULL), generator=torch.Generator().manual_seed(SEED))
VAL_IDX, TRAIN_IDX = _perm[:VALIDATION_SIZE], _perm[VALIDATION_SIZE:]

_LOADERS_CACHE: Dict[str, Tuple[DataLoader, DataLoader, DataLoader]] = {}


def get_loaders(classification: str) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Returns (train, validation, test) loaders with labels mapped to the chosen classification."""
    if classification not in _LOADERS_CACHE:
        def labels(y: torch.Tensor) -> torch.Tensor:
            return ORIGINAL_TO_SIMPLIFIED[y] if classification == "simplified" else y

        train_ds = TensorDataset(X_TRAIN_FULL[TRAIN_IDX], labels(Y_TRAIN_FULL[TRAIN_IDX]))
        val_ds = TensorDataset(X_TRAIN_FULL[VAL_IDX], labels(Y_TRAIN_FULL[VAL_IDX]))
        test_ds = TensorDataset(X_TEST, labels(Y_TEST))
        _LOADERS_CACHE[classification] = (
            # drop_last avoids a final batch of size 1, which breaks BatchNorm1d in training mode.
            DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
            DataLoader(val_ds, batch_size=1000, shuffle=False),
            DataLoader(test_ds, batch_size=1000, shuffle=False),
        )
    return _LOADERS_CACHE[classification]


print(f"Train: {len(TRAIN_IDX)} | Validation: {len(VAL_IDX)} | Test: {len(X_TEST)}")

Train: 55000 | Validation: 5000 | Test: 10000


## 2. Arquitecturas

El modelo es siempre un `nn.Sequential` que empieza en `nn.Flatten()` y usa **solo** `nn.Linear`,
`nn.ReLU`, `nn.BatchNorm1d` y `nn.Dropout`. Cada capa oculta es
`Linear → [BatchNorm1d] → ReLU → Dropout` (el `BatchNorm1d` solo está si se habilita;
`Dropout(p=0)` equivale a la identidad, así que `p` se puede cambiar sin reconstruir la red).

- **Rectangular:** `L` capas ocultas con el mismo número de neuronas (por defecto `L = 2`, 512).
- **Convolucional (embudo):** interpolación lineal entre 784 y la salida `n_out`:

$$\text{neurons}(i) = \operatorname{round}\left(784 + (n_{out} - 784)\,\frac{i}{L+1}\right),\quad i = 1..L$$

Con `L = 2` y 10 salidas: 784 → 526 → 268 → 10.

In [3]:
def funnel_sizes(hidden_layers: int, n_out: int, n_in: int = INPUT_SIZE) -> List[int]:
    """Neurons per hidden layer, linearly interpolated between n_in and n_out."""
    return [round(n_in + (n_out - n_in) * i / (hidden_layers + 1)) for i in range(1, hidden_layers + 1)]


def default_optimizations() -> dict:
    return {
        "optimizer": "adam",            # "adam" | "sgd" (SGD uses momentum 0.9)
        "learning_rate": 1e-3,
        "weight_decay": 0.0,
        "dropout": 0.0,                 # probability p of every Dropout layer
        "batch_norm": False,
        "early_stopping_patience": 0,   # 0 = disabled
    }


def build_model(neurons_per_layer: List[int], n_out: int, batch_norm: bool = False,
                dropout: float = 0.0) -> nn.Sequential:
    layers: List[nn.Module] = [nn.Flatten()]
    in_features = INPUT_SIZE
    for width in neurons_per_layer:
        layers.append(nn.Linear(in_features, width))
        if batch_norm:
            layers.append(nn.BatchNorm1d(width))
        layers.append(nn.ReLU())
        layers.append(nn.Dropout(dropout))
        in_features = width
    layers.append(nn.Linear(in_features, n_out))
    return nn.Sequential(*layers)


class Network:
    """A model plus everything needed to describe, train and save it."""

    def __init__(self, model_type: str, classification: str, neurons_per_layer: List[int],
                 optimizations: Optional[dict] = None, accumulated_epochs: int = 0) -> None:
        self.model_type = model_type
        self.classification = classification
        self.neurons_per_layer = list(neurons_per_layer)
        self.optimizations = {**default_optimizations(), **(optimizations or {})}
        self.accumulated_epochs = accumulated_epochs
        self.train_losses: List[float] = []
        self.val_losses: List[float] = []
        self.model = build_model(
            self.neurons_per_layer, NUM_OUTPUTS[classification],
            batch_norm=self.optimizations["batch_norm"], dropout=self.optimizations["dropout"],
        ).to(DEVICE)

    @classmethod
    def create(cls, model_type: str, classification: str, hidden_layers: int = 2,
               neurons: int = 512, optimizations: Optional[dict] = None) -> "Network":
        if model_type == "convolutional":
            sizes = funnel_sizes(hidden_layers, NUM_OUTPUTS[classification])
        else:
            sizes = [neurons] * hidden_layers
        return cls(model_type, classification, sizes, optimizations)

    @property
    def hidden_layers(self) -> int:
        return len(self.neurons_per_layer)

    def set_dropout(self, p: float) -> None:
        self.optimizations["dropout"] = p
        for module in self.model:
            if isinstance(module, nn.Dropout):
                module.p = p

    def hyperparameters(self) -> dict:
        return {
            "model_type": self.model_type,
            "hidden_layers": self.hidden_layers,
            "neurons_per_layer": list(self.neurons_per_layer),
            "accumulated_epochs": self.accumulated_epochs,
            "classification": self.classification,
            "optimizations": copy.deepcopy(self.optimizations),
        }

    def describe(self) -> None:
        print("\n--- Red actual ---")
        print(f"Tipo:                {self.model_type}")
        print(f"Clasificación:       {self.classification} ({NUM_OUTPUTS[self.classification]} clases)")
        print(f"Capas ocultas:       {self.hidden_layers}")
        print(f"Neuronas por capa:   {self.neurons_per_layer}")
        print(f"Épocas acumuladas:   {self.accumulated_epochs}")
        print(f"Optimizaciones:      {self.optimizations}")
        print(f"Parámetros totales:  {sum(p.numel() for p in self.model.parameters()):,}")


# Sanity check of the funnel formula (default L = 2, 10 classes -> 526 and 268).
assert funnel_sizes(2, 10) == [526, 268], funnel_sizes(2, 10)

## 3. Entrenamiento y evaluación

- **Entrenar** usa `CrossEntropyLoss` y registra la pérdida de entrenamiento y de validación
  de cada época. Si se entrena de nuevo, las épocas se **acumulan**.
- Las **optimizaciones** (optimizador, *learning rate*, *weight decay*, *dropout*,
  *batch normalization* y *early stopping*) se activan desde el menú.
- **Probar** calcula la matriz de confusión sobre el conjunto de prueba y, por clase,
  *precision* y *recall*, además del *accuracy* global.

In [4]:
def make_optimizer(model: nn.Module, opt: dict) -> torch.optim.Optimizer:
    if opt["optimizer"] == "sgd":
        return torch.optim.SGD(model.parameters(), lr=opt["learning_rate"], momentum=0.9,
                               weight_decay=opt["weight_decay"])
    return torch.optim.Adam(model.parameters(), lr=opt["learning_rate"], weight_decay=opt["weight_decay"])


@torch.no_grad()
def evaluate_loss(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> Tuple[float, float]:
    """Returns (mean loss, accuracy) over a loader, with the model in eval mode."""
    model.eval()
    total_loss, correct, count = 0.0, 0, 0
    for images, targets in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        logits = model(images)
        total_loss += criterion(logits, targets).item() * len(targets)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        count += len(targets)
    return total_loss / count, correct / count


def train_network(network: Network, epochs: int) -> None:
    """Trains for up to `epochs` epochs; honours early stopping when enabled."""
    train_loader, val_loader, _ = get_loaders(network.classification)
    opt = network.optimizations
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(network.model, opt)
    patience = opt["early_stopping_patience"]
    best_val, best_state, bad_epochs = float("inf"), None, 0

    for _ in range(epochs):
        network.model.train()
        running, seen = 0.0, 0
        for images, targets in train_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(network.model(images), targets)
            loss.backward()
            optimizer.step()
            running += loss.item() * len(targets)
            seen += len(targets)

        train_loss = running / seen
        val_loss, val_acc = evaluate_loss(network.model, val_loader, criterion)
        network.train_losses.append(train_loss)
        network.val_losses.append(val_loss)
        network.accumulated_epochs += 1
        print(f"Epoch {network.accumulated_epochs:3d} | train loss {train_loss:.4f} | "
              f"val loss {val_loss:.4f} | val acc {val_acc * 100:.2f}%")

        if patience > 0:
            if val_loss < best_val - 1e-4:
                best_val, bad_epochs = val_loss, 0
                best_state = copy.deepcopy(network.model.state_dict())
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"Early stopping: no validation improvement in {patience} epochs.")
                    break

    if best_state is not None:
        network.model.load_state_dict(best_state)


def plot_loss(network: Network) -> None:
    epochs = range(1, len(network.train_losses) + 1)
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, network.train_losses, marker="o", label="Entrenamiento")
    plt.plot(epochs, network.val_losses, marker="s", label="Validación")
    plt.xlabel("Época (de esta sesión)")
    plt.ylabel("Pérdida (cross-entropy)")
    plt.title("Pérdida por época")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def predict(model: nn.Module, loader: DataLoader) -> Tuple[torch.Tensor, torch.Tensor]:
    model.eval()
    all_true, all_pred = [], []
    for images, targets in loader:
        pred = model(images.to(DEVICE)).argmax(dim=1).cpu()
        all_true.append(targets)
        all_pred.append(pred)
    return torch.cat(all_true), torch.cat(all_pred)


def confusion_matrix(y_true: torch.Tensor, y_pred: torch.Tensor, n_classes: int) -> torch.Tensor:
    """Rows = true class, columns = predicted class."""
    return torch.bincount(y_true * n_classes + y_pred, minlength=n_classes ** 2).reshape(n_classes, n_classes)


def compute_metrics(cm: torch.Tensor) -> Dict[str, torch.Tensor]:
    true_positives = cm.diag().float()
    precision = true_positives / cm.sum(dim=0).clamp(min=1)   # TP / predicted as class
    recall = true_positives / cm.sum(dim=1).clamp(min=1)      # TP / really of class
    accuracy = true_positives.sum() / cm.sum()
    return {"precision": precision, "recall": recall, "accuracy": accuracy}


def test_network(network: Network) -> float:
    _, _, test_loader = get_loaders(network.classification)
    names = CLASS_NAMES[network.classification]
    y_true, y_pred = predict(network.model, test_loader)
    cm = confusion_matrix(y_true, y_pred, len(names))
    metrics = compute_metrics(cm)

    print(f"\nAccuracy en prueba: {metrics['accuracy'].item() * 100:.2f}%")
    print(f"{'Clase':<14}{'Precision':>10}{'Recall':>10}")
    for i, name in enumerate(names):
        print(f"{name:<14}{metrics['precision'][i].item():>10.4f}{metrics['recall'][i].item():>10.4f}")
    print(f"{'Promedio':<14}{metrics['precision'].mean().item():>10.4f}{metrics['recall'].mean().item():>10.4f}")

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.matshow(cm.numpy(), cmap="Blues")
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, str(cm[i, j].item()), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=8)
    ax.set_xticks(range(len(names)), names, rotation=45, ha="left")
    ax.set_yticks(range(len(names)), names)
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    ax.xaxis.set_label_position("top")
    ax.set_title("Matriz de confusión (conjunto de prueba)", pad=60)
    plt.tight_layout()
    plt.show()
    return metrics["accuracy"].item()

## 4. Persistencia (`*.pth`)

Cada archivo se guarda con `torch.save({"state_dict": ..., "hyperparameters": {...}}, path)`.
El modelo guardado es un `nn.Sequential` (Flatten + Linear/ReLU/BatchNorm1d/Dropout) que recibe
píxeles en `[0, 1]`. Orden de salida: 10 clases = orden nativo de Fashion MNIST;
4 clases = `0` Top, `1` Bottom, `2` Footwear, `3` Bag.

Nombre sugerido: `pytorch_fashion_N_<original|simplified>_<rectangular|convolutional>.pth`.

In [5]:
def default_filename(network: Network) -> str:
    return f"pytorch_fashion_{NOTEBOOK_NUMBER}_{network.classification}_{network.model_type}.pth"


def save_network(network: Network, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "state_dict": network.model.state_dict(),
        "hyperparameters": network.hyperparameters(),
    }, path)


def load_network(path: Path) -> Network:
    checkpoint = torch.load(Path(path), map_location=DEVICE, weights_only=True)
    hp = checkpoint["hyperparameters"]
    network = Network(
        model_type=hp["model_type"],
        classification=hp["classification"],
        neurons_per_layer=hp["neurons_per_layer"],
        optimizations=hp.get("optimizations"),
        accumulated_epochs=hp["accumulated_epochs"],
    )
    network.model.load_state_dict(checkpoint["state_dict"])   # strict: shapes must match
    return network

## 5. Menú interactivo

Primero se **crea** una red nueva o se **carga** de un archivo `.pth`. Luego el menú se repite
después de cada acción, así que se puede seguir entrenando. Presiona **Enter** para aceptar el
valor por defecto que aparece entre corchetes.

In [6]:
def ask(prompt: str, cast, default=None, valid=None):
    """Asks until the answer parses with `cast` and satisfies `valid`; Enter returns `default`."""
    suffix = f" [{default}]" if default is not None else ""
    while True:
        raw = input(f"{prompt}{suffix}: ").strip()
        if not raw and default is not None:
            return default
        try:
            value = cast(raw)
        except ValueError:
            print("Entrada inválida.")
            continue
        if valid is not None and not valid(value):
            print("Valor fuera de rango.")
            continue
        return value


def create_network_interactive() -> Network:
    print("\n--- Crear nueva red ---")
    classification = "simplified" if ask(
        "Clasificación (1 = original 10 clases, 2 = simplificada 4 clases)", int, 1,
        lambda v: v in (1, 2)) == 2 else "original"
    model_type = "convolutional" if ask(
        "Tipo de red (1 = rectangular, 2 = convolucional/embudo)", int, 1,
        lambda v: v in (1, 2)) == 2 else "rectangular"
    hidden_layers = ask("Número de capas ocultas", int, 2, lambda v: v >= 1)
    neurons = 512
    if model_type == "rectangular":
        neurons = ask("Neuronas por capa", int, 512, lambda v: v >= 1)
    network = Network.create(model_type, classification, hidden_layers, neurons)
    print(f"Red creada: 784 -> {' -> '.join(map(str, network.neurons_per_layer))} -> "
          f"{NUM_OUTPUTS[classification]}")
    return network


def optimization_menu(network: Network) -> Network:
    """Edits the optimizations. Toggling BatchNorm rebuilds the network (weights are reset)."""
    opt = network.optimizations
    while True:
        print("\n--- Optimizaciones ---")
        print(f"1. Optimizador:             {opt['optimizer']}")
        print(f"2. Learning rate:           {opt['learning_rate']}")
        print(f"3. Weight decay:            {opt['weight_decay']}")
        print(f"4. Dropout (p):             {opt['dropout']}")
        print(f"5. Batch normalization:     {'activada' if opt['batch_norm'] else 'desactivada'}")
        print(f"6. Early stopping (paciencia, 0 = off): {opt['early_stopping_patience']}")
        print("7. Volver")
        choice = input("Opción: ").strip()
        if choice == "1":
            opt["optimizer"] = "sgd" if ask("Optimizador (1 = Adam, 2 = SGD con momentum)", int, 1,
                                            lambda v: v in (1, 2)) == 2 else "adam"
        elif choice == "2":
            opt["learning_rate"] = ask("Learning rate", float, opt["learning_rate"], lambda v: v > 0)
        elif choice == "3":
            opt["weight_decay"] = ask("Weight decay", float, opt["weight_decay"], lambda v: v >= 0)
        elif choice == "4":
            network.set_dropout(ask("Probabilidad de dropout (0 a 0.9)", float, opt["dropout"],
                                    lambda v: 0 <= v < 1))
        elif choice == "5":
            answer = input("Cambiar BatchNorm reconstruye la red y REINICIA los pesos y las épocas. "
                           "¿Continuar? (s/n): ").strip().lower()
            if answer == "s":
                new_opt = {**opt, "batch_norm": not opt["batch_norm"]}
                network = Network(network.model_type, network.classification,
                                  network.neurons_per_layer, new_opt)
                opt = network.optimizations
                print("Red reconstruida con los pesos reiniciados.")
        elif choice == "6":
            opt["early_stopping_patience"] = ask("Paciencia (épocas sin mejora)", int,
                                                 opt["early_stopping_patience"], lambda v: v >= 0)
        elif choice == "7":
            return network
        else:
            print("Opción inválida.")


def main_menu() -> None:
    print("========================================")
    print("   FASHION MNIST CON PYTORCH (Tarea 3)  ")
    print("========================================")
    network: Optional[Network] = None
    while network is None:
        print("\n1. Crear una red nueva\n2. Cargar una red desde un archivo .pth")
        choice = input("Opción: ").strip()
        if choice == "1":
            network = create_network_interactive()
        elif choice == "2":
            path = input("Ruta del archivo .pth: ").strip().strip('"')
            try:
                network = load_network(Path(path).expanduser())
                print("Red cargada correctamente.")
            except Exception as error:
                print(f"No se pudo cargar la red: {error}")
        else:
            print("Opción inválida.")

    while True:
        network.describe()
        print("\n1. Entrenar\n2. Probar\n3. Guardar red\n4. Optimizaciones\n5. Salir")
        choice = input("Opción: ").strip()
        if choice == "1":
            epochs = ask("Número de épocas", int, 10, lambda v: v >= 1)
            network.train_losses.clear()
            network.val_losses.clear()
            train_network(network, epochs)
            plot_loss(network)
        elif choice == "2":
            test_network(network)
        elif choice == "3":
            path = ask("Ruta de guardado", str, default_filename(network), lambda v: len(v) > 0)
            try:
                save_network(network, Path(path).expanduser())
                print(f"Red guardada en {path}")
            except OSError as error:
                print(f"No se pudo guardar: {error}")
        elif choice == "4":
            network = optimization_menu(network)
        elif choice == "5":
            print("Finalizando programa.")
            break
        else:
            print("Opción inválida.")

### Utilidad para generar las cuatro redes entregables

La función de abajo entrena y guarda con el nombre correcto las cuatro redes
(rectangular/convolucional × original/simplificada), con `L = 2`, *dropout* 0.2 y *early stopping*.
**No se ejecuta sola**: llámala desde una celda aparte (`train_all_deliverables()`) o arma cada red
a mano desde el menú.

In [7]:
def train_all_deliverables(epochs: int = 15, output_dir: Path = Path(".")) -> None:
    configs = [("original", "rectangular"), ("original", "convolutional"),
               ("simplified", "rectangular"), ("simplified", "convolutional")]
    for classification, model_type in configs:
        network = Network.create(
            model_type, classification, hidden_layers=2, neurons=512,
            optimizations={**default_optimizations(), "dropout": 0.2, "early_stopping_patience": 3},
        )
        print(f"\n=== {classification} / {model_type}: {network.neurons_per_layer} ===")
        train_network(network, epochs)
        accuracy = test_network(network)
        save_network(network, Path(output_dir) / default_filename(network))
        print(f"Saved {default_filename(network)} (test accuracy {accuracy * 100:.2f}%)")

## Menú

In [ ]:
main_menu()

   FASHION MNIST CON PYTORCH (Tarea 3)  

1. Crear una red nueva
2. Cargar una red desde un archivo .pth


In [ ]:
import torch
from pathlib import Path
from torch import nn
from torchvision import datasets, transforms

test = datasets.FashionMNIST("data", train=False, transform=transforms.ToTensor())
X = torch.stack([test[i][0] for i in range(len(test))]); y = test.targets
simp = torch.tensor([0, 1, 0, 1, 0, 2, 0, 2, 3, 2])

def rebuild(sd):
    ws = [v for k, v in sd.items() if k.endswith(".weight") and v.dim() == 2]
    has_bn = any("running_mean" in k for k in sd)
    layers = [nn.Flatten()]
    for i, w in enumerate(ws):
        layers.append(nn.Linear(w.shape[1], w.shape[0]))
        if i < len(ws) - 1:
            if has_bn:
                layers.append(nn.BatchNorm1d(w.shape[0]))
            layers += [nn.ReLU(), nn.Dropout(0.0)]
    return nn.Sequential(*layers)

for p in sorted(Path(".").glob("pytorch_fashion_1_*.pth")):
    ck = torch.load(p, map_location="cpu", weights_only=True)
    m = rebuild(ck["state_dict"])
    m.load_state_dict(ck["state_dict"])      # strict: falla si algo no calza
    m.eval()
    with torch.no_grad():
        pred = m(X).argmax(1)
    target = simp[y] if "simplified" in p.name else y
    print(p.name, f"{(pred == target).float().mean().item():.4f}")

pytorch_fashion_1_original_convolutional.pth 0.8860
pytorch_fashion_1_original_rectangular.pth 0.8832
pytorch_fashion_1_simplified_convolutional.pth 0.9686
pytorch_fashion_1_simplified_rectangular.pth 0.9722
